# Regression Models with Keras

## Introduction
Despite the popularity of more powerful libraries such as PyToch and TensorFlow, they are not easy to use and have a steep learning curve. So, for people who are just starting to learn deep learning, there is no better library to use other than the Keras library. 

Keras is a high-level API for building deep learning models. It has gained favor for its ease of use and syntactic simplicity facilitating fast development. As you will see in this lab and the other labs in this course, building a very complex deep learning network can be achieved with Keras with only few lines of code. You will appreciate Keras even more, once you learn how to build deep models using PyTorch and TensorFlow in the other courses.


## Objectives for this Notebook    
* How to use the Keras library to build a regression model
* Download and clean the data set
* Build a neural network
* Train and test the network     



#### Import Dataset

In [7]:
import numpy as np
import keras
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests


In [8]:
def download(url,fileName):
    response = requests.get(url)
    if(response.status_code == 200 ):
        with open(fileName, "wb") as f:
            f.write(response.content)

In [10]:
filepath='https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0101EN/labs/data/concrete_data.csv'
file_name = 'concrete_data.csv'
download(filepath,file_name)

In [11]:
df = pd.read_csv(file_name)
df.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [12]:
# check how many data we have
df.shape

(1030, 9)

In [13]:
#check missing values
df.describe()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [14]:
df.isnull().sum()

Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
Strength              0
dtype: int64

#### Split data into predictors and target


In [28]:
#predictors will be the all the columns except the concreate strength


predictors = df.drop(columns={'Strength'})
target = df['Strength']
predictors_columns = predictors.columns



In [25]:
print("predictors:\n", predictors.head())
print("\ntarget:\n", target.head())

predictors:
    Cement  Blast Furnace Slag  Fly Ash  Water  Superplasticizer  \
0   540.0                 0.0      0.0  162.0               2.5   
1   540.0                 0.0      0.0  162.0               2.5   
2   332.5               142.5      0.0  228.0               0.0   
3   332.5               142.5      0.0  228.0               0.0   
4   198.6               132.4      0.0  192.0               0.0   

   Coarse Aggregate  Fine Aggregate  Age  
0            1040.0           676.0   28  
1            1055.0           676.0   28  
2             932.0           594.0  270  
3             932.0           594.0  365  
4             978.4           825.5  360  

target:
 0    79.99
1    61.89
2    40.27
3    41.05
4    44.30
Name: Strength, dtype: float64


#### Standardization the Data

In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
predictors_scaled = pd.DataFrame(scaler.fit_transform(predictors),columns=predictors_columns)

print("Scaled predictors:\n", predictors_scaled.head())

Scaled predictors:
      Cement  Blast Furnace Slag   Fly Ash     Water  Superplasticizer  \
0  2.477915           -0.856888 -0.847144 -0.916764         -0.620448   
1  2.477915           -0.856888 -0.847144 -0.916764         -0.620448   
2  0.491425            0.795526 -0.847144  2.175461         -1.039143   
3  0.491425            0.795526 -0.847144  2.175461         -1.039143   
4 -0.790459            0.678408 -0.847144  0.488793         -1.039143   

   Coarse Aggregate  Fine Aggregate       Age  
0          0.863154       -1.217670 -0.279733  
1          1.056164       -1.217670 -0.279733  
2         -0.526517       -2.240917  3.553066  
3         -0.526517       -2.240917  5.057677  
4          0.070527        0.647884  4.978487  


In [42]:
# number of columns
n_cols = predictors_scaled.shape[1]
n_cols 

8

### Import Keras Package

In [43]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input

### Build Neural Network

In [46]:
def regression_model():
    
    model = Sequential()
    model.add(Input(shape=(n_cols,)))
    model.add(Dense(50,activation='relu'))
    model.add(Dense(50,activation='relu'))
    model.add(Dense(1))
    
    #compile the model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

## Train and Test the Network


In [47]:
#Build the model
model = regression_model()

Next, we will train and test the model at the same time using the *fit* method. We will leave out 30% of the data for validation and we will train the model for 100 epochs.


In [48]:
model.fit(predictors_scaled,target,validation_split=0.3,epochs=100, verbose=2)

Epoch 1/100
23/23 - 2s - 68ms/step - loss: 1668.3635 - val_loss: 1152.5658
Epoch 2/100
23/23 - 0s - 6ms/step - loss: 1537.0236 - val_loss: 1035.2765
Epoch 3/100
23/23 - 0s - 6ms/step - loss: 1337.2964 - val_loss: 862.5869
Epoch 4/100
23/23 - 0s - 6ms/step - loss: 1040.8108 - val_loss: 638.7996
Epoch 5/100
23/23 - 0s - 5ms/step - loss: 686.5903 - val_loss: 414.0443
Epoch 6/100
23/23 - 0s - 5ms/step - loss: 400.3535 - val_loss: 254.0700
Epoch 7/100
23/23 - 0s - 5ms/step - loss: 263.7910 - val_loss: 187.1994
Epoch 8/100
23/23 - 0s - 8ms/step - loss: 224.1950 - val_loss: 167.7390
Epoch 9/100
23/23 - 0s - 7ms/step - loss: 208.9272 - val_loss: 159.8141
Epoch 10/100
23/23 - 0s - 8ms/step - loss: 199.9518 - val_loss: 158.3642
Epoch 11/100
23/23 - 0s - 7ms/step - loss: 193.3493 - val_loss: 157.3512
Epoch 12/100
23/23 - 0s - 5ms/step - loss: 187.2962 - val_loss: 156.0791
Epoch 13/100
23/23 - 0s - 6ms/step - loss: 182.2327 - val_loss: 153.1874
Epoch 14/100
23/23 - 0s - 6ms/step - loss: 178.9400 -